In [4]:
import io.github.francescodonnini.csv.CsvIssueApi
import io.github.francescodonnini.csv.CsvJavaClassApi
import io.github.francescodonnini.csv.CsvJavaMethodApi
import io.github.francescodonnini.csv.CsvReleaseApi
import org.eclipse.jgit.api.Git
import org.eclipse.jgit.revwalk.RevCommit
import org.eclipse.jgit.storage.file.FileRepositoryBuilder
import kotlin.io.path.Path

val project = "SYNCOPE"
val gitPath = "/home/francesco/Documents/Università/ISW/Progetto/60/sources/${project.toLowerCase()}/.git"
val path = "/home/francesco/Documents/Università/ISW/Progetto/60/data/$project"
val releaseApi = CsvReleaseApi(path + "/releases.csv")
val git = Git(FileRepositoryBuilder()
                .setGitDir(Path(gitPath).toFile())
                .build())
val commits = ArrayList<RevCommit>();
git.log().call().forEach { commits.add(it) }
val issueApi = CsvIssueApi(path + "/issues.csv", releaseApi.getLocal(), commits)

In [2]:
val tags = git.tagList().call()
    .map { Pair(it.name, git.getRepository().parseCommit(it.objectId).authorIdent.`when`) }
    .map { it -> Pair(it.first.replace("refs/tags/syncope-", ""), java.text.SimpleDateFormat("yyyy-MM-dd").format(it.second)) }
    .sortedBy { it -> it.second }
    .toList()
val releases = releaseApi.getLocal()
    .sortedBy { it -> it.releaseDate }

id,name,releaseDate
12320044,1.0.0-incubating,2012-08-06
12322540,1.0.1-incubating,2012-08-29
12323346,1.0.3-incubating,2012-09-30
12323254,1.0.2-incubating,2012-10-02
12323474,1.0.4,2012-12-10
12323749,1.0.5,2013-01-23
12324001,1.0.6,2013-02-27
12324104,1.0.7,2013-03-26
12322504,1.1.0,2013-04-05
12324279,1.0.8,2013-04-18


In [5]:
val issues = issueApi.getLocal()
    .filter { it.affectedVersions.isNotEmpty() }
    .sortedBy { it.created }
    .toList()

In [6]:
val issuesFrame = dataFrameOf(
    "key" to issues.map { it.key },
    "created" to issues.map { it.created },
    "affectedVersions" to issues.map { it.affectedVersions.map { it -> if (it == null) "null" else it.name }.joinToString(",") },
    "commits" to issues.map { it.commits.map { it -> it.name.substring(0..5) }.joinToString(",") }
)
issuesFrame


key,created,affectedVersions,commits
SYNCOPE-187,2012-08-14T08:35:21,"1.0.0-incubating,1.1.0",945a21
SYNCOPE-196,2012-09-03T10:32:43,1.0.0-incubating,6bfba4
SYNCOPE-267,2013-01-08T10:00:41,"1.0.4,1.0.5,1.1.0","5d58e0,415c57"
SYNCOPE-269,2013-01-10T11:08:23,"1.0.4,1.1.0",4fefb4
SYNCOPE-274,2013-01-15T11:35:22,1.0.4,72e4d9
SYNCOPE-294,2013-01-28T11:30:59,1.0.5,968bd6
SYNCOPE-301,2013-01-30T11:27:32,"1.0.5,1.1.0",e52201
SYNCOPE-308,2013-02-04T16:55:08,"1.0.5,1.1.0",cb0738
SYNCOPE-309,2013-02-04T17:18:28,"1.0.5,1.1.0","eb4f73,81501d"
SYNCOPE-310,2013-02-08T11:34:53,"1.0.5,1.1.0","9ffda2,3ef7ba"


In [7]:
import org.eclipse.jgit.diff.DiffFormatter
import org.eclipse.jgit.util.io.DisabledOutputStream

val df = DiffFormatter(DisabledOutputStream.INSTANCE)
df.setRepository(git.repository)
df.setDetectRenames(true)
val issue = issues[0]
println(issue)

Issue[affectedVersions=[(12320044, 1.0.0-incubating, 2012-08-06), (12322504, 1.1.0, 2013-04-05)], created=2012-08-14T08:35:21, fixVersion=(12322540, 1.0.1-incubating, 2012-08-29), openingVersion=(12322540, 1.0.1-incubating, 2012-08-29), commits=[commit 945a21ed1a2e43f80ae49c5b72bc3f251536a9f2 1344953620 -----sp], key=SYNCOPE-187, project=Syncope]


In [21]:
val commit = issue.commits[0]
println(commit.parents[0].name)
println(commit.name)
val diffList = df.scan(commit.tree, commit.parents[0].tree)

f7b46d67fb5979c5175185b20e3ca54a44fe2497
945a21ed1a2e43f80ae49c5b72bc3f251536a9f2


In [22]:
for (d in diffList) {
    println(d)
}

DiffEntry[MODIFY console/src/main/java/org/apache/syncope/console/pages/panels/ResultSetPanel.java]
DiffEntry[MODIFY console/src/main/java/org/apache/syncope/console/rest/UserRestClient.java]


In [23]:
for (diff in diffList) {
    val path = diff.newPath
    println(path)
    val editList = df.toFileHeader(diff).toEditList()
    for (edit in editList) {
        println(edit)
        println("A\t${edit.beginA} ${edit.endA}")
        println("B\t${edit.beginB} ${edit.endB}")
    }
}

console/src/main/java/org/apache/syncope/console/pages/panels/ResultSetPanel.java
REPLACE(469-471,469-470)
A	469 471
B	469 470
console/src/main/java/org/apache/syncope/console/rest/UserRestClient.java
REPLACE(148-149,148-149)
A	148 149
B	148 149
REPLACE(154-155,154-155)
A	154 155
B	154 155
REPLACE(162-163,162-163)
A	162 163
B	162 163
